# Integrating Cricsheet Data with Cricinfo

This notebook focuses on parsing ball-by-ball cricket data from Cricsheet (JSON format) and preparing it for integration with other datasets.

## Objectives:
1. Load JSON files from the `data` folder.
2. Parse match information and delivery details.
3. Flatten the nested structure into a pandas DataFrame.
4. Perform initial data cleaning and type optimization.

In [13]:
import pandas as pd
import json
import os
from glob import glob
from datetime import datetime
import numpy as np

## Data Loading and Parsing Functions

In [14]:
def extract_match_data(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    match_info = data.get('info', {})
    match_id = os.path.basename(json_file).split('.')[0]
    
    deliveries_list = []
    
    for inning_idx, inning in enumerate(data.get('innings', [])):
        team_name = inning.get('team')
        overs = inning.get('overs', [])
        
        for over_data in overs:
            over_num = over_data.get('over')
            deliveries = over_data.get('deliveries', [])
            
            for ball_idx, delivery in enumerate(deliveries):
                ball_data = {
                    'match_id': match_id,
                    'season': match_info.get('season'),
                    'start_date': match_info.get('dates', [None])[0],
                    'venue': match_info.get('venue'),
                    'innings': inning_idx + 1,
                    'batting_team': team_name,
                    'over': over_num,
                    'ball': ball_idx + 1,
                    'batter': delivery.get('batter'),
                    'bowler': delivery.get('bowler'),
                    'non_striker': delivery.get('non_striker'),
                    'runs_off_bat': delivery.get('runs', {}).get('batter', 0),
                    'extras': delivery.get('runs', {}).get('extras', 0),
                    'total_runs': delivery.get('runs', {}).get('total', 0)
                }
                
                # Handle wickets
                wickets = delivery.get('wickets', [])
                if wickets:
                    ball_data['is_wicket'] = 1
                    ball_data['player_out'] = wickets[0].get('player_out')
                    ball_data['wicket_type'] = wickets[0].get('kind')
                else:
                    ball_data['is_wicket'] = 0
                    ball_data['player_out'] = None
                    ball_data['wicket_type'] = None
                
                # Handle extras breakdown
                extra_details = delivery.get('extras', {})
                ball_data['wides'] = extra_details.get('wides', 0)
                ball_data['noballs'] = extra_details.get('noballs', 0)
                ball_data['byes'] = extra_details.get('byes', 0)
                ball_data['legbyes'] = extra_details.get('legbyes', 0)
                ball_data['penalty'] = extra_details.get('penalty', 0)
                
                deliveries_list.append(ball_data)
                
    return deliveries_list

## Process All Files

We will load all JSON files from the `data` folder and consolidate them into a single DataFrame.

In [15]:
data_dir = '../data'
json_files = glob(os.path.join(data_dir, '*.json'))

print(f"Found {len(json_files)} JSON files.")

all_data = []
for i, file in enumerate(json_files):
    if i % 100 == 0:
        print(f"Processing file {i}/{len(json_files)}...")
    all_data.extend(extract_match_data(file))

df = pd.DataFrame(all_data)
print("Data processing complete.")

Found 3193 JSON files.
Processing file 0/3193...
Processing file 100/3193...
Processing file 200/3193...
Processing file 300/3193...
Processing file 400/3193...
Processing file 500/3193...
Processing file 600/3193...
Processing file 700/3193...
Processing file 800/3193...
Processing file 900/3193...
Processing file 1000/3193...
Processing file 1100/3193...
Processing file 1200/3193...
Processing file 1300/3193...
Processing file 1400/3193...
Processing file 1500/3193...
Processing file 1600/3193...
Processing file 1700/3193...
Processing file 1800/3193...
Processing file 1900/3193...
Processing file 2000/3193...
Processing file 2100/3193...
Processing file 2200/3193...
Processing file 2300/3193...
Processing file 2400/3193...
Processing file 2500/3193...
Processing file 2600/3193...
Processing file 2700/3193...
Processing file 2800/3193...
Processing file 2900/3193...
Processing file 3000/3193...
Processing file 3100/3193...
Data processing complete.


## Data Cleaning and Optimization

We'll convert column types to more efficient formats as seen in other project notebooks.

In [16]:
df['start_date'] = pd.to_datetime(df['start_date'])
df['match_id'] = df['match_id'].astype('category')
df['season'] = df['season'].astype('category')
df['venue'] = df['venue'].astype('category')
df['batting_team'] = df['batting_team'].astype('category')
df['batter'] = df['batter'].astype('category')
df['bowler'] = df['bowler'].astype('category')
df['is_wicket'] = df['is_wicket'].astype('int8')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720644 entries, 0 to 720643
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   match_id      720644 non-null  category      
 1   season        720644 non-null  category      
 2   start_date    720644 non-null  datetime64[ns]
 3   venue         720644 non-null  category      
 4   innings       720644 non-null  int64         
 5   batting_team  720644 non-null  category      
 6   over          720644 non-null  int64         
 7   ball          720644 non-null  int64         
 8   batter        720644 non-null  category      
 9   bowler        720644 non-null  category      
 10  non_striker   720644 non-null  object        
 11  runs_off_bat  720644 non-null  int64         
 12  extras        720644 non-null  int64         
 13  total_runs    720644 non-null  int64         
 14  is_wicket     720644 non-null  int8          
 15  player_out    410

In [17]:
df.head(10)

,match_id,season,start_date,venue,innings,batting_team,over,ball,batter,bowler,...,extras,total_runs,is_wicket,player_out,wicket_type,wides,noballs,byes,legbyes,penalty
0,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,1,M Klinger,SL Malinga,...,0,0,0,None,None,0,0,0,0,0
1,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,2,M Klinger,SL Malinga,...,0,1,0,None,None,0,0,0,0,0
2,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,3,AJ Finch,SL Malinga,...,0,0,0,None,None,0,0,0,0,0
3,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,4,AJ Finch,SL Malinga,...,0,4,0,None,None,0,0,0,0,0
4,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,5,AJ Finch,SL Malinga,...,1,1,0,None,None,0,0,0,1,0
5,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,0,6,M Klinger,SL Malinga,...,0,1,0,None,None,0,0,0,0,0
6,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,1,1,M Klinger,KMDN Kulasekara,...,1,1,0,None,None,1,0,0,0,0
7,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,1,2,M Klinger,KMDN Kulasekara,...,0,1,0,None,None,0,0,0,0,0
8,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,1,3,AJ Finch,KMDN Kulasekara,...,0,2,0,None,None,0,0,0,0,0
9,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",1,Australia,1,4,AJ Finch,KMDN Kulasekara,...,0,2,0,None,None,0,0,0,0,0


## Match Metadata

We have extracted the match metadata from the Readme.txt found in the cricsheet zip file using a python script, thereby generating a different dataset named 'match_metadata.csv'. 

Now, let's start analyzing the match_metadata.csv 

In [18]:
# Load match metadata and consolidated delivery data
metadata_df = pd.read_csv('../data/match_metadata.csv')
delivery_df = pd.read_csv('../data/consolidated_t20_data.csv')

# Ensure match_id type consistency for merging
metadata_df['match_id'] = metadata_df['match_id'].astype(str)
delivery_df['match_id'] = delivery_df['match_id'].astype(str)

print(f"Metadata shape: {metadata_df.shape}")
print(f"Delivery data shape: {delivery_df.shape}")
metadata_df.head()

Metadata shape: (3194, 6)
Delivery data shape: (720644, 27)


,date,team_type,match_type,gender,match_id,teams
0,2026-03-04,international,T20,male,1512771,South Africa vs New Zealand
1,2026-03-01,international,T20,male,1525162,Hong Kong vs Kuwait
2,2026-03-01,international,T20,male,1512770,West Indies vs India
3,2026-03-01,international,T20,male,1512769,Zimbabwe vs South Africa
4,2026-02-28,international,T20,male,1523005,Japan vs Bahrain


In [19]:
# Merge the dataframes on match_id
merged_df = pd.merge(delivery_df, metadata_df, on='match_id', how='inner')

# Remove all NaN values
merged_df.dropna(inplace=True)

print(f"Merged dataframe shape: {merged_df.shape}")
merged_df.head()

Merged dataframe shape: (39554, 32)


,match_id,season,start_date,venue,city,gender_x,match_type_x,team_type_x,innings,batting_team,...,wides,noballs,byes,legbyes,penalty,date,team_type_y,match_type_y,gender_y,teams
16,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",Victoria,male,T20,international,1,Australia,...,0,0,0,0,0,2017-02-19,international,T20,male,Australia vs Sri Lanka
41,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",Victoria,male,T20,international,1,Australia,...,0,0,0,0,0,2017-02-19,international,T20,male,Australia vs Sri Lanka
81,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",Victoria,male,T20,international,1,Australia,...,0,0,0,0,0,2017-02-19,international,T20,male,Australia vs Sri Lanka
88,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",Victoria,male,T20,international,1,Australia,...,0,0,0,0,0,2017-02-19,international,T20,male,Australia vs Sri Lanka
98,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",Victoria,male,T20,international,1,Australia,...,0,0,0,0,0,2017-02-19,international,T20,male,Australia vs Sri Lanka


## Checking for Data Mismatches

We will now compare the shared columns from original delivery data and match metadata to ensure consistency.

In [20]:
# Ensure both date columns are in datetime format
merged_df['start_date'] = pd.to_datetime(merged_df['start_date'])
merged_df['date'] = pd.to_datetime(merged_df['date'])

# Pairs of columns to compare for mismatches
compare_pairs = [
    ('start_date', 'date'),
    ('gender_x', 'gender_y'),
    ('match_type_x', 'match_type_y'),
    ('team_type_x', 'team_type_y')
]

for col1, col2 in compare_pairs:
    mismatches = merged_df[merged_df[col1] != merged_df[col2]]
    num_mismatches = len(mismatches)
    print(f"--- Comparing {col1} vs {col2} ---")
    print(f"Number of rows with mismatches: {num_mismatches}")
    
    if num_mismatches > 0:
        mismatched_ids = mismatches[['match_id', col1, col2]].drop_duplicates()
        print(f"Mismatched Match IDs and values:")
        print(mismatched_ids)
    print("\n")

--- Comparing start_date vs date ---
Number of rows with mismatches: 0


--- Comparing gender_x vs gender_y ---
Number of rows with mismatches: 0


--- Comparing match_type_x vs match_type_y ---
Number of rows with mismatches: 0


--- Comparing team_type_x vs team_type_y ---
Number of rows with mismatches: 0




## Column Cleaning and Renaming

Based on the mismatch check, we will drop duplicate columns and rename the primary ones to be more concise.

In [21]:
# Define columns to drop (the _y versions and the redundant date column)
cols_to_drop = ['date', 'gender_y', 'match_type_y', 'team_type_y']
merged_df.drop(columns=cols_to_drop, inplace=True)

# Rename columns
rename_dict = {
    'gender_x': 'gender',
    'match_type_x': 'match_type',
    'team_type_x': 'team_type'
}
merged_df.rename(columns=rename_dict, inplace=True)

In [22]:
# Reorder columns logically (Metadata first, then Delivery details)
cols = list(merged_df.columns)
metadata_cols = ['match_id', 'season', 'start_date', 'venue', 'gender', 'match_type', 'team_type', 'teams']
print(metadata_cols)
delivery_cols = [c for c in cols if c not in metadata_cols]
print(delivery_cols)
new_order = metadata_cols + delivery_cols
print(new_order)
merged_df = merged_df[new_order]

print(f"Final columns: {merged_df.columns.tolist()}")
merged_df.head()

['match_id', 'season', 'start_date', 'venue', 'gender', 'match_type', 'team_type', 'teams']
['city', 'innings', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_off_bat', 'extras', 'total_runs', 'is_wicket', 'player_out', 'wicket_type', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']
['match_id', 'season', 'start_date', 'venue', 'gender', 'match_type', 'team_type', 'teams', 'city', 'innings', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_off_bat', 'extras', 'total_runs', 'is_wicket', 'player_out', 'wicket_type', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']
Final columns: ['match_id', 'season', 'start_date', 'venue', 'gender', 'match_type', 'team_type', 'teams', 'city', 'innings', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_off_bat', 'extras', 'total_runs', 'is_wicket', 'player_out', 'wicket_type', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']


,match_id,season,start_date,venue,gender,match_type,team_type,teams,city,innings,...,extras,total_runs,is_wicket,player_out,wicket_type,wides,noballs,byes,legbyes,penalty
16,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",male,T20,international,Australia vs Sri Lanka,Victoria,1,...,0,0,1,AJ Finch,caught and bowled,0,0,0,0,0
41,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",male,T20,international,Australia vs Sri Lanka,Victoria,1,...,0,0,1,BR Dunk,bowled,0,0,0,0,0
81,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",male,T20,international,Australia vs Sri Lanka,Victoria,1,...,0,0,1,M Klinger,caught,0,0,0,0,0
88,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",male,T20,international,Australia vs Sri Lanka,Victoria,1,...,0,0,1,TM Head,caught,0,0,0,0,0
98,1001351,2016/17,2017-02-19,"Simonds Stadium, South Geelong",male,T20,international,Australia vs Sri Lanka,Victoria,1,...,0,0,1,AJ Turner,caught,0,0,0,0,0


In [25]:
# export dataframe to csv

merged_df.to_csv("../data/merged.csv", index=False)